# A training cohort with a study-level split

A model trained on spatial data should be tested on data from experiments it has not seen. If sections from the
same experiment end up in both the training and the test fold, the test score measures how well the model
recognises that experiment, not how well it generalises. The notebook builds a small cohort for training, splits
it so that each experiment stays in one fold, reads one table per fold and records the split so that it can be
reproduced.

It uses the steps from the tutorials, mainly {doc}`../tutorials/03_cohorts_and_manifests`, and adds
`split()` and the `TableAdapter` interface. The cohort is small (five Visium sections of human pancreatic
tumours, of which only the tables are read), so it runs on a laptop.

In [ ]:
import tempfile
from pathlib import Path
from typing import Any

import sddb
from sddb import SplitManifest, SpatialDataCohort, TableAdapter

cat = sddb.Catalog(version="2026-10-28")
work = Path(tempfile.mkdtemp())  # the files this notebook writes go here

## The cohort

Five Visium sections of human pancreatic ductal adenocarcinoma from three GEO series. The `collections` column
names the series of each section.

In [ ]:
cohort = cat.cohort(
    uids=[
        "qEP9pd4LlyKeufe20000",
        "URKDREjVhAsRwwF10000",
        "Mc06lamGrOTYHhNF0000",
        "0877Bxvk1AFUpoD90000",
        "StreGVL1Uyj2EeoM0000",
    ]
)
cohort.to_df()[["uid", "study_id", "collections", "n_obs", "n_features"]]

## Which key keeps an experiment together

`split(by=...)` assigns whole groups to folds, and the default group is `study_id`. For GEO data the `study_id`
is the sample accession, so every section is its own group, and sections of one series can land in different
folds:

In [ ]:
def series_per_fold(folds: dict[str, SpatialDataCohort]) -> dict[str, list[str]]:
    """The GEO series in each fold, from the collections column."""
    return {fold: sorted({c for cs in members.members["collections"] for c in cs}) for fold, members in folds.items()}


by_sample = cohort.split(by="study_id", train=0.6, val=0.2, test=0.2, seed=0)
series_per_fold(by_sample)

A series in two folds means that the test fold holds sections of an experiment the model was trained on.
`collections` is a list column, so `split` cannot group by it directly. Add a column with the series and split a
cohort built from those rows. That cohort has no link to the catalog file, so freeze the original cohort to record
where the data came from (see the end of this notebook).

In [ ]:
rows = cohort.members.assign(series=[cs[0] for cs in cohort.members["collections"]])
grouped = SpatialDataCohort(rows)
folds = grouped.split(by="series", train=0.4, val=0.3, test=0.3, seed=0)
{fold: (len(members), sorted(set(members.members["series"]))) for fold, members in folds.items()}

Fold sizes are counted in groups, not in sections, so with three series each fold gets one series, whatever the
ratios. With more series the ratios apply as you would expect. For sources other than GEO, check which column
groups an experiment (for example `collections` or `doi`) before you split.

## A table adapter

`to_anndata(adapter=...)` passes each member's table through an adapter before the join. An adapter is any object
with the methods of the `TableAdapter` protocol, so it can check a member, drop it, or map its table into the
space your model expects. The adapter below keeps the spots under the tissue and drops a section with fewer than 500 such
spots.

In [ ]:
class InTissueSpots:
    """Keep spots under the tissue; drop sections with fewer than ``min_spots`` of them."""

    name = "in-tissue-spots"
    version = "0.1.0"

    def __init__(self, min_spots: int = 200) -> None:
        self.min_spots = min_spots

    def requirements(self) -> dict[str, Any]:
        return {}  # read the whole table

    def validate_meta(self, row) -> list[str]:
        return [] if row["technology"] == "Visium" else [f"not Visium: {row['technology']}"]

    def validate(self, adata, row) -> list[str]:
        if "in_tissue" not in adata.obs:
            return ["no obs['in_tissue']"]
        n = int((adata.obs["in_tissue"] == 1).sum())
        return [] if n >= self.min_spots else [f"only {n} spots under the tissue"]

    def build(self, adata, row):
        return adata[adata.obs["in_tissue"] == 1].copy()


adapter = InTissueSpots(min_spots=500)
isinstance(adapter, TableAdapter)

## One table per fold

Each fold is a cohort, so `to_anndata` reads its tables and joins them. `gene_key="symbol"` puts the genes of
every member under their symbols.

In [ ]:
tables = {fold: members.to_anndata(gene_key="symbol", adapter=adapter) for fold, members in folds.items()}
for fold, adata in tables.items():
    print(f"{fold:5s} {adata.n_obs:5d} spots x {adata.n_vars} genes from {adata.obs['uid'].nunique()} sections")

Sections that the adapter dropped are listed in `uns["sddb"]["dropped"]` with the reason:

In [ ]:
{fold: adata.uns.get("sddb", {}).get("dropped", {}) for fold, adata in tables.items()}

The folds have different numbers of genes, because the series store different gene lists (17943, 36601 and 19144 genes). A model needs the
same features in every fold, so keep the genes that all folds share, in one fixed order:

In [ ]:
shared = sorted(set.intersection(*(set(adata.var_names) for adata in tables.values())))
tables = {fold: adata[:, shared].copy() for fold, adata in tables.items()}
print(len(shared), "genes in every fold")
{fold: adata.shape for fold, adata in tables.items()}

`X` holds the counts and `obs["uid"]` the section of each spot, which is what a training loop needs.

In [ ]:
train = tables["train"]
print(type(train.X).__name__, train.X.shape, f"{train.X.sum():.0f} counts")
train.obs["uid"].value_counts()

## Record the split

Every fold carries the same `SplitManifest`. It records the group column, the seed, the ratios, the fold of every
uid and the hash of the cohort it was made from. Write it next to the model, together with the manifest of the
original cohort, which records the catalog release.

In [ ]:
split = folds["train"].split_manifest
split.write(work / "split.json")
cohort.freeze().write(work / "cohort.json")

again = SplitManifest.load(work / "split.json")
print(again.by, again.seed, again.ratios)
print("same assignments:", again.assignments == split.assignments)
print("made from the grouped cohort:", again.parent_hash == grouped.freeze().hash)

`split()` is deterministic. The same cohort, column, ratios and seed give the same folds on any machine, because
the groups are sorted before a seeded shuffle.

In [ ]:
repeat = grouped.split(by="series", train=0.4, val=0.3, test=0.3, seed=0)
repeat["train"].split_manifest.assignments == split.assignments